In [118]:
!pip install langchain
!pip install langchain-community
!pip install google-generativeai
!pip install langchain-google-genai
!pip install pypdf
!pip install sentence-transformers
!pip install SpeechRecognition pyaudio
!pip install pyttsx3
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install groq
!pip install ipywidgets


In [119]:
import os

folders = ["data", "faiss_db", "memory"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

memory_file = "memory/long_term_memory.json"
if not os.path.exists(memory_file):
    import json
    with open(memory_file, "w") as f:
        json.dump([], f)

print("Folder structure ready!")
print(os.listdir("."))

Folder structure ready!
['architecture.txt', 'data', 'docs', 'faiss_db', 'memory', 'ragproject1.ipynb']


In [120]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


documents = []
for file in os.listdir("data"):
    if file.endswith(".pdf"):
        pdf_path = os.path.join("data", file)
        loader = PyPDFLoader(pdf_path)
        documents.extend(loader.load())
print("Documents loaded:", len(documents))


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(documents)
print("Chunks created:", len(chunks))


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


db = FAISS.from_documents(chunks, embeddings)
db.save_local("faiss_db")
print("Database Created Successfully")

Documents loaded: 341
Chunks created: 1406


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Database Created Successfully


In [128]:
import os
import json
import torch
from google import genai
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from datetime import datetime
import speech_recognition as sr


GEMINI_KEY_1 = "AQ.Ab8RN6IbwAxW9VdMYQ5hWzh0y108XL14IMSo2PCkwgAcpzolcQ"
GEMINI_KEY_2 = "AQ.Ab8RN6KlptFhxkh5PRzhDqxcegzTyNto-w_2fN0ygxw4_nWQ1w"


GROQ_API_KEY = "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"


gemini_client_1 = genai.Client(api_key=GEMINI_KEY_1)
gemini_client_2 = genai.Client(api_key=GEMINI_KEY_2)

from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)


def ask_gemini(client, prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text


def ask_groq(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

device = "cuda" if torch.cuda.is_available() else "cpu"
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True}
)


db = FAISS.load_local("faiss_db", embeddings, allow_dangerous_deserialization=True)


conversation_history = []


MEMORY_FILE = "memory/long_term_memory.json"

def load_long_term_memory():
    with open(MEMORY_FILE, "r") as f:
        return json.load(f)

def save_long_term_memory(memories):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memories, f, indent=2)

def add_to_long_term_memory(question, answer):
    memories = load_long_term_memory()
    
    
    keywords = [w for w in question.lower().split() 
                if len(w) > 4 and w not in 
                ["what", "how", "when", "where", "which", "would", "could", "should", "about", "their"]]
    
    memories.append({
        "timestamp": datetime.now().isoformat(),
        "question": question,
        "summary": answer[:300],
        "topics": keywords[:5]      
    })
    
    if len(memories) > 100:          
        memories = memories[-100:]
    save_long_term_memory(memories)


def listen():
    r = sr.Recognizer()

    with sr.Microphone() as source:
        print("🎤 Listening...")
        audio = r.listen(source)

    try:
        text = r.recognize_google(audio)
        print("You said:", text)
        return text

    except Exception as e:
        print("Error:", e)
        return None
def get_relevant_memories(question, top_n=3):
    memories = load_long_term_memory()
    if not memories:
        return ""

    import datetime

    scored = []

    for m in memories:
        q_overlap = len(set(question.lower().split()) & set(m["question"].lower().split()))

        
        try:
            days_old = (datetime.datetime.now() - datetime.datetime.fromisoformat(m["timestamp"])).days
        except:
            days_old = 999

        recency_score = max(0, 10 - days_old)

        score = q_overlap + recency_score * 0.2

        scored.append((score, m))

    scored.sort(reverse=True, key=lambda x: x[0])

    top = [m for score, m in scored[:top_n] if score > 0]

    text = "Relevant past conversations:\n"
    for m in top:
       text += f"- [{m['timestamp'][:10]}] You asked: {m['question']}\n"

    return text
def find_exact_time(question):
    memories = load_long_term_memory()

    question_words = set(question.lower().split())

    best_match = None
    best_score = 0

    for m in memories:
        memory_words = set(m["question"].lower().split())

        # overlap score
        score = len(question_words & memory_words)

        if score > best_score:
            best_score = score
            best_match = m

    if best_match and best_score > 0:
        return best_match["timestamp"]

    return None
print(f"Loaded! Device: {device}")
print(f"Long-term memories: {len(load_long_term_memory())}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded! Device: cpu
Long-term memories: 28


In [122]:
from click import prompt


def chat_with_andrew(question):
    global conversation_history

    
    
    docs = db.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content[:400] for doc in docs])


   
    recent_history = conversation_history[-4:]
    history_text = ""
    for turn in recent_history:
        history_text += f"User: {turn['user']}\nAndrew Ng: {turn['assistant']}\n\n"

    
    long_term_text = get_relevant_memories(question)
    exact_time = find_exact_time(question)

    if exact_time:
     time_info = f"Exact timestamp for this topic: {exact_time}"
    else:
     time_info = "No exact timestamp found."
    
    prompt = f"""
You are Andrew Ng answering questions in the style of his public lectures, interviews, writings, and courses.

Stay in character as an educator, AI researcher, entrepreneur, and builder of machine learning systems.

========================
STYLE
========================
- Prioritize intuition before technical details.
- Explain why something matters before explaining how it works.
- Use clear, simple language.
- Break complex ideas into smaller concepts.
- Use concrete examples and everyday analogies.
- Be thoughtful, humble, and practical.
- Maintain an optimistic but realistic view of AI.

========================
TEACHING PRINCIPLES
========================
- Focus on helping the learner understand rather than impressing them.
- Connect ideas to real-world applications.
- Emphasize experimentation and implementation.
- Encourage curiosity and continuous learning.

========================
CRITICAL RETRIEVAL RULES
========================
- Use retrieved context as the PRIMARY source of truth.
- If context contains the answer, rely on it.
- If not, use general knowledge while staying in character.
- NEVER invent facts not supported by context or well-known public information.

========================
MEMORY RULES (VERY IMPORTANT)
========================
- You are given past conversations as memory.
- Use memory ONLY to maintain continuity.
- Do NOT guess missing details from memory.
- If memory is unclear, say you are not sure.

========================
TIME / TIMESTAMP RULES (CRITICAL FIX)
========================
- Current date: {datetime.now().strftime('%Y-%m-%d')}

- You may be given a variable called `time_info`.

RULES:
1. If `time_info` contains a valid timestamp → you may mention it.
2. If `time_info` says "No exact timestamp found" → explicitly say:
   "I don’t have an exact timestamp for this."
3. NEVER guess or approximate time.
4. NEVER say "recently" or "earlier today" when timestamp is unknown.

========================
VOICE STYLE
========================
- Use natural teaching speech:
  - "The key insight is..."
  - "One way to think about this is..."
  - "Let me break this down..."

========================
REASONING RULES
========================
- Do NOT fabricate timeline reasoning.
- Do NOT assume order unless explicitly provided in memory timestamps.
- If uncertain, explicitly say uncertainty.

========================
INPUT CONTEXTS
========================

Long-term memory:
{long_term_text}

Time info:
{time_info}

Recent conversation:
{history_text}

Retrieved context:
{context}

========================
QUESTION
========================
{question}

========================
RESPONSE (Andrew Ng style):
"""


    

    providers = [
      ("Gemini Key 1", lambda: ask_gemini(gemini_client_1, prompt)),
      ("Gemini Key 2", lambda: ask_gemini(gemini_client_2, prompt)),
      ("Groq", lambda: ask_groq(prompt)),
]

    answer = ""

    for provider_name, provider_func in providers:

      try:
        print(f"Trying {provider_name}...")

        answer = provider_func()

        print(f"Success using {provider_name}")
        break

      except Exception as e:
        print(f"{provider_name} failed: {e}")

    if not answer:
     raise Exception("All LLM providers failed")
 
   
    conversation_history.append({
        "user": question,
        "assistant": answer
    })

    add_to_long_term_memory(question, answer)

    return answer

print("Chat function ready!")

Chat function ready!


In [129]:
sample_questions = [
    "What is the most important skill for someone starting in machine learning?",
    "How do you think about bias and variance tradeoff?",
    "What advice would you give to students who want to break into AI?",
    "Can you explain how neural networks learn?",
    "What do you think is the biggest challenge in AI today?",
    "How should someone structure their learning path for deep learning?",
    "What is your view on AI replacing jobs?",
    "How do you approach debugging a machine learning model?",
    "What made you start Coursera and DeepLearning.AI?",
    "How do you stay updated with the fast pace of AI research?"
]

print("Generating 10 sample conversations...\n")

sample_output = []
for i, q in enumerate(sample_questions, 1):
    print(f"Generating {i}/10...")
    answer = chat_with_andrew(q)
    sample_output.append({
        "conversation": i,
        "user": q,
        "andrew_ng": answer
    })
    import time
    time.sleep(3)  

#
import json
with open("docs/sample_conversations.json", "w") as f:
    json.dump(sample_output, f, indent=2)

print("\nSample conversations saved to docs/sample_conversations.json")

Generating 10 sample conversations...

Generating 1/10...
Trying Gemini Key 1...
Success using Gemini Key 1
Generating 2/10...
Trying Gemini Key 1...
Success using Gemini Key 1
Generating 3/10...
Trying Gemini Key 1...
Success using Gemini Key 1
Generating 4/10...
Trying Gemini Key 1...
Success using Gemini Key 1
Generating 5/10...
Trying Gemini Key 1...
Success using Gemini Key 1
Generating 6/10...
Trying Gemini Key 1...
Success using Gemini Key 1
Generating 7/10...
Trying Gemini Key 1...
Gemini Key 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Trying Gemini Key 2...
Gemini Key 2 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Trying Groq...
Success using Groq
Generating 8/10.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [ ]:
import pyttsx3

engine = pyttsx3.init()

def speak(text):
    engine.say(text)
    engine.runAndWait()

In [ ]:
import speech_recognition as sr

def listen():

    recognizer = sr.Recognizer()

    with sr.Microphone() as source:

        print(" Listening...")

        recognizer.adjust_for_ambient_noise(source)

        audio = recognizer.listen(source)

    try:

        text = recognizer.recognize_google(audio)

        print("You said:", text)

        return text

    except Exception as e:

        print("Could not understand audio")

        return None

In [ ]:
import ipywidgets as widgets
from IPython.display import display


chat_messages = []
last_answer = ""


def on_voice(b):

    global last_answer

    question = listen()

    if not question:
        return

    
    chat_messages.append({
        "user": question,
        "assistant": "⏳ Thinking..."
    })

    update_chat()

    
    answer = chat_with_andrew(question)

    last_answer = answer

    
    chat_messages[-1]["assistant"] = answer

    update_chat()

    
    speak(answer)

chat_history = widgets.HTML(
    value="",
    layout=widgets.Layout(
        border="1px solid #ddd",
        height="500px",
        width="100%",
        overflow="auto"
    )
)
memory_output = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #ccc',
        height='250px',
        overflow_y='scroll',
        width='100%'
    )
)


user_input = widgets.Textarea(
    placeholder="Ask Andrew Ng anything...",
    layout=widgets.Layout(
        width="85%",
        height="80px"
    )
)


send_button = widgets.Button(
    description="Send",
    button_style="success",
    icon="paper-plane"
)
memory_button = widgets.Button(
    description="🧠 View Memory",
    button_style="info"
)

clear_memory_button = widgets.Button(
    description="🗑 Clear Memory",
    button_style="danger"
)
clear_button = widgets.Button(
    description="Clear",
    button_style="warning",
    icon="trash"
)

voice_button = widgets.Button(
    description="🎤 Speak",
    button_style="primary",
    icon="microphone"
)

voice_button.on_click(on_voice)

def update_chat():

    html = """
    <div style="
        height:480px;
        overflow-y:auto;
        padding:15px;
        font-family:Arial, sans-serif;
        background-color:#fafafa;
    ">
    """

    for msg in chat_messages:

        html += f"""
        <div style="
            text-align:right;
            margin-bottom:10px;
        ">
            <div style="
                display:inline-block;
                background:#DCF8C6;
                padding:10px;
                border-radius:12px;
                max-width:80%;
            ">
                <b>🧑 You</b><br>
                {msg['user']}
            </div>
        </div>
        """

        html += f"""
        <div style="
            text-align:left;
            margin-bottom:20px;
        ">
            <div style="
                display:inline-block;
                background:#F1F1F1;
                padding:10px;
                border-radius:12px;
                max-width:80%;
            ">
                <b>🤖 Andrew Ng</b><br>
                {msg['assistant']}
            </div>
        </div>
        """

    html += "</div>"

    chat_history.value = html



def on_send(b):

    global last_answer

    question = user_input.value.strip()

    if not question:
        return

    send_button.disabled = True
    send_button.description = "Thinking..."

    try:

        answer = chat_with_andrew(question)

        last_answer = answer

        chat_messages.append({
            "user": question,
            "assistant": answer
        })

        update_chat()

    except Exception as e:

        chat_messages.append({
            "user": question,
            "assistant": f"Error: {str(e)}"
        })

        update_chat()

    finally:

        user_input.value = ""
        send_button.disabled = False
        send_button.description = "Send"



def clear_chat(b):

    global chat_messages

    chat_messages = []

    update_chat()


def view_memory(b):

    memories = load_long_term_memory()

    with memory_output:

        memory_output.clear_output()

        print(f"Total Memories: {len(memories)}\n")

        if not memories:
            print("No memories stored.")
            return

        for i, m in enumerate(reversed(memories[-20:]), start=1):

            print("=" * 70)

            print(f"Memory #{i}")

            print("Date:", m["timestamp"][:19])

            print("Question:")
            print(m["question"])

            print()

            print("Topics:")
            print(", ".join(m["topics"]))

            print()

            print("Summary:")
            print(m["summary"])

            print()
            
def clear_memory(b):

    save_long_term_memory([])

    with memory_output:

        memory_output.clear_output()

        print("✅ All long-term memories deleted.")



def submit_text(change):

    if change["new"].endswith("\n"):

        user_input.value = user_input.value.rstrip("\n")

        on_send(None)



send_button.on_click(on_send)
clear_button.on_click(clear_chat)
voice_button.on_click(on_voice)
memory_button.on_click(view_memory)

clear_memory_button.on_click(clear_memory)
user_input.observe(submit_text, names="value")



title = widgets.HTML(
    value="""
    <h2 style='color:#2E86C1'>
        🤖 Andrew Ng Digital Twin
    </h2>
    """
)


ui = widgets.VBox([
    title,
     
    chat_history,

    widgets.HBox([
        user_input,
        send_button
    ]),

    widgets.HBox([
        voice_button,
        clear_button,
        memory_button,
        clear_memory_button
    ]),

    memory_output
])

display(ui)

update_chat()